In [31]:
# All of the imports we require for the project 
import glob 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import chi2_contingency

In [7]:
# read the data from the species CSV file into pandas dataframe

species_info_df = pd.read_csv('species_info.csv')


# print the head of each the species dataframe
species_info_df.head()


,category,scientific_name,common_names,conservation_status
0,Mammal,Clethrionomys gapperi gapperi,Gapper's Red-Backed Vole,NaN
1,Mammal,Bos bison,"American Bison, Bison",NaN
2,Mammal,Bos taurus,"Aurochs, Aurochs, Domestic Cattle (Feral), Dom...",NaN
3,Mammal,Ovis aries,"Domestic Sheep, Mouflon, Red Sheep, Sheep (Feral)",NaN
4,Mammal,Cervus elaphus,Wapiti Or Elk,NaN


In [8]:
 # read the data from the observations CSV file into pandas dataframe

observations_df = pd.read_csv('observations.csv')
observations_df.head()

,scientific_name,park_name,observations
0,Vicia benghalensis,Great Smoky Mountains National Park,68
1,Neovison vison,Great Smoky Mountains National Park,77
2,Prunus subcordata,Yosemite National Park,138
3,Abutilon theophrasti,Bryce National Park,84
4,Githopsis specularioides,Great Smoky Mountains National Park,85


In [9]:
# use the info() method to get a concise summary of both dataframes 

species_info_df.info()
observations_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5824 entries, 0 to 5823
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   category             5824 non-null   object
 1   scientific_name      5824 non-null   object
 2   common_names         5824 non-null   object
 3   conservation_status  191 non-null    object
dtypes: object(4)
memory usage: 182.1+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23296 entries, 0 to 23295
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   scientific_name  23296 non-null  object
 1   park_name        23296 non-null  object
 2   observations     23296 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 546.1+ KB


In [14]:
# use describe() method to get a statistical summary of both dataframes

species_info_df.describe()
print("\n")
print(observations_df.describe())



       observations
count  23296.000000
mean     142.287904
std       69.890532
min        9.000000
25%       86.000000
50%      124.000000
75%      195.000000
max      321.000000


In [15]:
# display column names for both dataframes
print(species_info_df.columns)
print(observations_df.columns)

Index(['category', 'scientific_name', 'common_names', 'conservation_status'], dtype='object')
Index(['scientific_name', 'park_name', 'observations'], dtype='object')


In [19]:
# get the sum of null values in each dataframe
print(species_info_df.isnull().sum())
print(observations_df.isnull().sum())


category                  0
scientific_name           0
common_names              0
conservation_status    5633
dtype: int64
scientific_name    0
park_name          0
observations       0
dtype: int64


In [42]:
# get the number of duplicate rows in each dataframe
print(species_info_df.duplicated().sum())
print("The number of duplicate rows in observations was initially 15")

# show the duplicate rows in the observations dataframe
observations_df[observations_df.duplicated()]

# drop the duplicate rows in the observations dataframe
observations_df = observations_df.drop_duplicates()

# get the number of duplicate rows in observations dataframe after dropping duplicates

print(observations_df.duplicated().sum())

0
The number of duplicate rows in observations was initially 15
0


In [ ]:
# investigate categorical columns in the species_info_df dataframe
print(species_info_df['category'].value_counts())

print(species_info_df['conservation_status'].value_counts())

# print the unique values in the category and conservation_status columns of the species_info_df dataframe
print(species_info_df['category'].unique())

print(species_info_df['conservation_status'].unique())


category
Vascular Plant       4470
Bird                  521
Nonvascular Plant     333
Mammal                214
Fish                  127
Amphibian              80
Reptile                79
Name: count, dtype: int64
conservation_status
Species of Concern    161
Endangered             16
Threatened             10
In Recovery             4
Name: count, dtype: int64
['Mammal' 'Bird' 'Reptile' 'Amphibian' 'Fish' 'Vascular Plant'
 'Nonvascular Plant']
[nan 'Species of Concern' 'Endangered' 'Threatened' 'In Recovery']


In [22]:
# investigate categorical columns in the observations_df dataframe
print(observations_df["park_name"].unique())

['Great Smoky Mountains National Park' 'Yosemite National Park'
 'Bryce National Park' 'Yellowstone National Park']


In [27]:
# hone in on the missing values in the conservation_status column of the species_info_df dataframe
print(species_info_df["conservation_status"].value_counts(dropna=False))

# display the rows where conservation_status is missing
print(species_info_df[species_info_df["conservation_status"].isnull()].head())

# couting the number of species in each category that are missing a conservation status
print(species_info_df[species_info_df["conservation_status"].isnull()]["category"].value_counts())

conservation_status
NaN                   5633
Species of Concern     161
Endangered              16
Threatened              10
In Recovery              4
Name: count, dtype: int64
  category                scientific_name  \
0   Mammal  Clethrionomys gapperi gapperi   
1   Mammal                      Bos bison   
2   Mammal                     Bos taurus   
3   Mammal                     Ovis aries   
4   Mammal                 Cervus elaphus   

                                        common_names conservation_status  
0                           Gapper's Red-Backed Vole                 NaN  
1                              American Bison, Bison                 NaN  
2  Aurochs, Aurochs, Domestic Cattle (Feral), Dom...                 NaN  
3  Domestic Sheep, Mouflon, Red Sheep, Sheep (Feral)                 NaN  
4                                      Wapiti Or Elk                 NaN  
category
Vascular Plant       4424
Bird                  442
Nonvascular Plant     328
Mammal     

In [29]:
# We want to calculate the proportion of species in each category that are missing a conservation status.

missing_status_by_category = species_info_df[species_info_df["conservation_status"].isnull()]["category"].value_counts()

total_by_category = species_info_df["category"].value_counts()

proportion_missing = (missing_status_by_category / total_by_category) * 100

print(proportion_missing)

category
Amphibian            91.250000
Bird                 84.836852
Fish                 91.338583
Mammal               82.242991
Nonvascular Plant    98.498498
Reptile              93.670886
Vascular Plant       98.970917
Name: count, dtype: float64


In [34]:
# create a contingency table to show the relationship between category and conservation_status missing values

contingency_table = pd.crosstab(species_info_df["category"], species_info_df["conservation_status"].isnull())

print(contingency_table)

# calculate the chi square statistic and p-value for the contingency table

chi2, p, dof, expected = chi2_contingency(contingency_table)

print("Chi-square statistic: ", np.round(chi2, 2))
print("p-value: ", np.round(p, 2))
print("expected frequencies: ", np.round(expected, 2))


conservation_status  False  True 
category                         
Amphibian                7     73
Bird                    79    442
Fish                    11    116
Mammal                  38    176
Nonvascular Plant        5    328
Reptile                  5     74
Vascular Plant          46   4424
Chi-square statistic:  469.51
p-value:  0.0
expected frequencies:  [[2.6200e+00 7.7380e+01]
 [1.7090e+01 5.0391e+02]
 [4.1700e+00 1.2283e+02]
 [7.0200e+00 2.0698e+02]
 [1.0920e+01 3.2208e+02]
 [2.5900e+00 7.6410e+01]
 [1.4660e+02 4.3234e+03]]
